# `examples/custom_scheduler` 命令行等价运行版

这个 Notebook 不再把 `examples/custom_scheduler/main.py` 重新拆开实现，而是直接在 Notebook 中以子进程方式执行原始 `.py` 文件：

```bash
python -u examples/custom_scheduler/main.py
```

这样运行路径、日志行为和你在 PowerShell 中直接执行脚本基本一致，能避免 Notebook 里直接调用 `sim.run()` 时日志不显示、输出被缓存或内核表现为“像卡住一样”的问题。

适用场景：

1. 先确认官方/项目已有样例能稳定跑通；
2. 保持 `.py` 样例作为唯一真实入口；
3. Notebook 只负责启动、显示日志和辅助说明；
4. 后续再逐步把结果导出和分析逻辑补进去。

## 1. 定位项目根目录

下面的代码会从当前 Notebook 所在目录开始，向上寻找同时包含 `sim/` 和 `examples/` 的目录，并把它作为 faas-sim 项目根目录。

In [1]:
from pathlib import Path
import sys
import os
import subprocess

current_dir = Path.cwd().resolve()

candidate_roots = [
    current_dir,
    current_dir.parent,
    current_dir.parent.parent,
    current_dir.parent.parent.parent,
]

PROJECT_ROOT = None
for root in candidate_roots:
    if (root / "sim").exists() and (root / "examples").exists():
        PROJECT_ROOT = root
        break

if PROJECT_ROOT is None:
    raise RuntimeError(
        "没有找到 faas-sim 项目根目录。请把 Notebook 放在项目根目录、examples/custom_scheduler/ "
        "或其相邻目录下运行。"
    )

script_path = PROJECT_ROOT / "examples" / "custom_scheduler" / "main.py"

print(f"当前 Notebook 工作目录：{current_dir}", flush=True)
print(f"faas-sim 项目根目录：{PROJECT_ROOT}", flush=True)
print(f"即将运行的样例脚本：{script_path}", flush=True)

if not script_path.exists():
    raise FileNotFoundError(f"找不到样例脚本：{script_path}")

当前 Notebook 工作目录：C:\Users\weew12\Downloads\faas-sim-master_内置依赖兼容性检查版\faas-sim-master\examples\custom_scheduler
faas-sim 项目根目录：C:\Users\weew12\Downloads\faas-sim-master_内置依赖兼容性检查版\faas-sim-master
即将运行的样例脚本：C:\Users\weew12\Downloads\faas-sim-master_内置依赖兼容性检查版\faas-sim-master\examples\custom_scheduler\main.py


## 2. 使用当前 Jupyter 内核对应的 Python 运行样例

这里使用 `sys.executable`，保证 Notebook 使用哪个 Python 内核，就用哪个 Python 来执行样例脚本。

`-u` 参数表示 unbuffered，作用是让 Python 日志和 print 输出尽快刷新出来，尽量接近你在命令行中看到的效果。

In [2]:
print("当前 Jupyter 内核 Python：", sys.executable, flush=True)
print("开始以命令行等价方式运行 custom_scheduler 样例。", flush=True)

当前 Jupyter 内核 Python： d:\miniconda3\python.exe
开始以命令行等价方式运行 custom_scheduler 样例。


## 3. 执行 `examples/custom_scheduler/main.py`

这一格会实时打印子进程输出。

如果 `.py` 脚本在 PowerShell 中能跑通，这一格通常也应该能跑通，并显示类似：

```text
INFO:sim.faassim:initializing simulation...
INFO:sim.faas.system:deploying function python-pi...
INFO:sim.faas.system:pod pod-python-pi-1 was scheduled to ...
INFO:sim.faassim:simulation ran ...
```

In [3]:
env = os.environ.copy()

# 确保子进程优先从项目根目录导入 sim、examples、ether、skippy、simpy 等本地包。
existing_pythonpath = env.get("PYTHONPATH", "")
env["PYTHONPATH"] = (
    str(PROJECT_ROOT)
    if not existing_pythonpath
    else str(PROJECT_ROOT) + os.pathsep + existing_pythonpath
)

cmd = [
    sys.executable,
    "-u",
    str(script_path),
]

print("执行命令：", " ".join(cmd), flush=True)
print("工作目录：", PROJECT_ROOT, flush=True)
print("开始输出子进程日志：", flush=True)

process = subprocess.Popen(
    cmd,
    cwd=str(PROJECT_ROOT),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

# 实时转发子进程输出。
for line in process.stdout:
    print(line, end="", flush=True)

return_code = process.wait()

print(f"\n子进程退出码：{return_code}", flush=True)

if return_code != 0:
    raise RuntimeError(f"custom_scheduler 样例运行失败，退出码：{return_code}")
else:
    print("custom_scheduler 样例运行完成。", flush=True)

执行命令： d:\miniconda3\python.exe -u C:\Users\weew12\Downloads\faas-sim-master_内置依赖兼容性检查版\faas-sim-master\examples\custom_scheduler\main.py
工作目录： C:\Users\weew12\Downloads\faas-sim-master_内置依赖兼容性检查版\faas-sim-master
开始输出子进程日志：
INFO:sim.faassim:initializing simulation, benchmark: ExampleBenchmark, topology nodes: 171
INFO:__main__:creating CustomScheduler
INFO:sim.faassim:starting resource monitor
INFO:sim.faassim:setting up benchmark
INFO:examples.basic.main:python-pi-cpu, latest, [ImageProperties(name='python-pi-cpu', size=58000000, tag='latest', arch='arm32'), ImageProperties(name='python-pi-cpu', size=58000000, tag='latest', arch='x86'), ImageProperties(name='python-pi-cpu', size=58000000, tag='latest', arch='aarch64')]
INFO:examples.basic.main:resnet50-inference-cpu, latest, [ImageProperties(name='resnet50-inference-cpu', size=56000000, tag='latest', arch='arm32'), ImageProperties(name='resnet50-inference-cpu', size=56000000, tag='latest', arch='x86'), ImageProperties(name='resnet50-in

## 4. 说明：为什么这个版本更稳定

之前 Notebook 版本是把 `.py` 文件逻辑拆到多个单元格中，然后在 Notebook 内部直接调用：

```python
sim.run()
```

这种方式理论上可以运行，但 Jupyter/IPython 对标准输出、日志 handler、当前工作目录、已加载模块缓存都有自己的管理方式，容易出现：

1. `logging.basicConfig()` 不生效；
2. 日志没有显示到输出区；
3. 单元格正在运行但没有可见输出；
4. 多次运行后模块缓存状态和命令行不完全一致；
5. Notebook 当前工作目录和命令行工作目录不同。

现在这个版本使用 `subprocess.Popen` 直接执行原始 `.py` 文件，等价于在命令行运行样例，更适合当前阶段“先把已有样例全部跑通”。

## 5. 后续建议

等所有样例都确认能通过 `.py` 入口稳定运行后，再考虑为每个样例补充第二类 Notebook：

1. **命令行等价运行版**：只负责跑原始 `.py`，确认样例能跑通；
2. **交互分析版**：重新创建 `Simulation` 对象，并在运行结束后提取 `metrics` DataFrame 做分析。

当前阶段建议优先使用命令行等价运行版，避免把“样例本身问题”和“Notebook 环境问题”混在一起。